# Fig. 6E: pseudotime transferred onto human nuclei, in the human sections

The human Slide-tags sections colored by the mouse pseudotime transferred through the SAMap alignment (the transfer
of record), on black: the secretory (decidualized) section and the menstrual section, one bare panel each, plus a
horizontal color key. Each phase is one physical section whose three 10x lanes share one coordinate frame, so they
are drawn together. Crops and rotations are those of the figure's first version (14 October). Nuclei without a valued
mouse neighbor are drawn in grey underneath; they are not imputed.

Display rules, as in the methods and the SAMap supplement: one fixed 0-1 viridis scale with values as stored,
never re-ranked. Every positioned nucleus is drawn, those flagged `embedding_outlier` included (decision of 1 Nov 2026:
the flag stays in the tables but no panel hides it; until then this panel left the flagged nuclei out).

**Inputs**: `joint.parquet` (rule `samap_figure_tables`: species, joint UMAP, 10x library and transferred values of every
nucleus) and the human fibroblast/decidual plot table of the release whose pseudotime was transferred (QC'd
Slide-tags positions, phase, `embedding_outlier`).
**Outputs**, in `OUT`: `<DATE>-Human-<phase>-transferred_pseudotime-spatial.{png,pdf,svg}` per phase (6 x 6 in, 300 dpi,
no text), `<DATE>-spatial_key_transferred_pseudotime.{png,pdf,svg}` and
`fig6e_sections_transferred_pseudotime.provenance.json` with the counts and median per section.

Fonts: Helvetica Neue where installed (macOS, the version of record), otherwise Helvetica, otherwise DejaVu Sans; the
provenance record names the font used.

**Running it.** Run all cells with this directory as the working directory: the shared modules next to the notebook
are imported from there. The cells above *Run* only define constants and functions. Paths are relative to the
repository root. Existing outputs are not overwritten unless `FORCE = True`.

The pipeline runs it with papermill (`workflow/rules/10c_samap_figures.smk`). By hand, from the repository root:

```bash
papermill notebooks/samap/fig6e_sections_transferred_pseudotime.ipynb \
    results/notebooks/samap/fig6e_sections_transferred_pseudotime.executed.ipynb \
    --cwd notebooks/samap -p FORCE True \
    -p HUMAN_TABLE results/plot_tables/<release>/human_fibdec_plot_table.parquet
```

Commit this notebook without outputs: the provenance record notes whether the notebook matches its commit.

In [ ]:
# Parameters. papermill -p NAME VALUE overrides them in a cell below this one.
# Paths are relative to the repository root; REPO = None finds the root with git from the working directory.
REPO = None
JOINT = "results/samap_figures/tables/joint.parquet"
HUMAN_TABLE = None  # results/plot_tables/<release>/human_fibdec_plot_table.parquet; the rule passes it
OUT = "results/figures/samap_main"
DATE = "20261027"  # file-name prefix of the panels
FORCE = False  # overwrite existing outputs
NOTEBOOK = "notebooks/samap/fig6e_sections_transferred_pseudotime.ipynb"  # this notebook, for the provenance record

In [ ]:
import datetime
import hashlib
import json
import logging
import platform
import subprocess
import sys
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display
from matplotlib import font_manager
from matplotlib.lines import Line2D

# This notebook's messages in the cell output; other libraries stay at WARNING.
logging.basicConfig(format="%(message)s", stream=sys.stdout, force=True)
LOG = logging.getLogger("fig6e")
LOG.setLevel(logging.INFO)

# The human plot table has no default: it is the release of record's (rule samap_fibdec_plot_table), which
# the rule passes; a run by hand passes it too (-p HUMAN_TABLE ...).
if HUMAN_TABLE is None:
    raise ValueError("set HUMAN_TABLE: -p HUMAN_TABLE results/plot_tables/<release>/human_fibdec_plot_table.parquet")

## Constants

In [ ]:
TRANSFERRED = "transferred_mm_dpt_pseudotime_ranknorm"  # the transfer of record, in joint.parquet

# Fixed 0-1 viridis with values as stored: a transfer that flattens the values shows as flat color.
CMAP = mpl.colormaps["viridis"]
NORM = mpl.colors.Normalize(0, 1)

# Helvetica Neue is a system font on macOS that matplotlib does not find on its own; elsewhere the family list
# falls through to DejaVu Sans.
FONT_FAMILY = ["Helvetica Neue", "Helvetica", "DejaVu Sans"]
MACOS_HELVETICA_NEUE = Path("/System/Library/Fonts/HelveticaNeue.ttc")
TEXT = "white"

# Per phase: crop (x and y limits in µm, None for the whole section), counter-clockwise rotation (degrees) and
# whether the rotated section is mirrored left-right about its center.
SECTIONS = {
    "secretory": {"limits": [(400, 5500), (750, 5250)], "rotation": -90, "mirror": True},
    "menstrual": {"limits": None, "rotation": 90, "mirror": True},
}

NO_VALUE = "#B0B0B0"  # human nuclei without a valued mouse SAMap neighbor
SPATIAL_IN = 6        # bare square panel, in inches
SPATIAL_DPI = 300
SPATIAL_DOT = 8       # pt^2 on the 6 in panel

## Figure

In [ ]:
def set_font() -> str:
    """Register Helvetica Neue on macOS and set the family list; returns the family that will be used."""
    if MACOS_HELVETICA_NEUE.exists():
        font_manager.fontManager.addfont(str(MACOS_HELVETICA_NEUE))
    mpl.rcParams.update({"font.family": FONT_FAMILY, "svg.fonttype": "none", "pdf.fonttype": 42})
    return font_manager.FontProperties(family=FONT_FAMILY).get_name()


def human_nuclei(joint: pd.DataFrame, human_table_path: Path) -> pd.DataFrame:
    """Human nuclei of the joint export with their position, phase and embedding_outlier flag from the release."""
    human_table = pd.read_parquet(human_table_path, columns=["x_um_qc", "y_um_qc", "timepoint", "embedding_outlier"])
    # timepoint comes from the release table; the joint export carries its own copy.
    human = joint[joint.species == "hs"].drop(columns=["timepoint"]).join(human_table, how="inner")
    if len(human) != (joint.species == "hs").sum():
        raise ValueError("human nuclei in the joint export are missing from the release table")
    return human


def find_repo(repo) -> Path:
    if repo is not None:
        return Path(repo)
    git_root = ["git", "rev-parse", "--show-toplevel"]
    return Path(subprocess.run(git_root, capture_output=True, text=True, check=True).stdout.strip())


def file_record(path: Path, role: str) -> dict:
    stat = path.stat()
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1 << 20), b""):
            digest.update(block)
    return {
        "path": str(path.resolve()),
        "role": role,
        "bytes": stat.st_size,
        "mtime": datetime.datetime.fromtimestamp(stat.st_mtime).isoformat(timespec="seconds"),
        "sha256": digest.hexdigest(),
    }


def git_revision(path: Path):
    """Commit of the checkout holding path, and whether path matches it; None outside git."""
    git = ["git", "-C", str(path.parent)]
    try:
        commit = subprocess.run(git + ["rev-parse", "HEAD"], capture_output=True, text=True, check=True)
        status = subprocess.run(git + ["status", "--porcelain", "--", path.name],
                                capture_output=True, text=True, check=True)
    except (subprocess.CalledProcessError, FileNotFoundError):
        return None
    return {"commit": commit.stdout.strip(), "notebook_matches_commit": status.stdout.strip() == ""}


def rotate(x, y, degrees, cx, cy):
    """Rotate points counter-clockwise by `degrees` about (cx, cy)."""
    if degrees % 360 == 0:
        return x, y
    t = np.deg2rad(degrees)
    return (x - cx) * np.cos(t) - (y - cy) * np.sin(t) + cx, (x - cx) * np.sin(t) + (y - cy) * np.cos(t) + cy


def section_frame(cells: pd.DataFrame, limits, degrees, mirror: bool) -> pd.DataFrame:
    """Nuclei inside the crop, with coordinates xr, yr (µm) rotated and, if mirror, mirrored left-right."""
    x, y = cells["x_um_qc"].to_numpy(float), cells["y_um_qc"].to_numpy(float)
    if limits is not None:
        (x0, x1), (y0, y1) = limits
        keep = (x >= x0) & (x <= x1) & (y >= y0) & (y <= y1)
        cells, x, y = cells[keep], x[keep], y[keep]
        cx, cy = (x0 + x1) / 2, (y0 + y1) / 2
    else:
        cx, cy = (x.min() + x.max()) / 2, (y.min() + y.max()) / 2
    xr, yr = rotate(x, y, degrees, cx, cy)
    if mirror:
        xr = 2 * cx - xr  # left-right, about the center the section was rotated around
    return cells.assign(xr=xr, yr=yr)


def marker_padding(span_x, span_y, dot_size) -> tuple:
    """Pad the data range by one marker radius (+2 px) so edge dots are not clipped on the bare panel."""
    radius_px = np.sqrt(dot_size / np.pi) / 72 * SPATIAL_DPI + 2
    side_px = SPATIAL_IN * SPATIAL_DPI

    def pad(span):
        return span * (radius_px / side_px) / (1 - 2 * radius_px / side_px) + 0.005 * span

    return pad(span_x), pad(span_y)


def spatial_panel(frame: pd.DataFrame, path: Path) -> None:
    """Bare black square: nuclei without a value underneath in grey, the rest by transferred pseudotime."""
    fig, ax = plt.subplots(figsize=(SPATIAL_IN, SPATIAL_IN), facecolor="black")
    ax.set_position([0, 0, 1, 1])
    ax.set_facecolor("black")
    pad_x, pad_y = marker_padding(np.ptp(frame.xr), np.ptp(frame.yr), SPATIAL_DOT)
    ax.set_xlim(frame.xr.min() - pad_x, frame.xr.max() + pad_x)
    ax.set_ylim(frame.yr.min() - pad_y, frame.yr.max() + pad_y)
    ax.set_aspect("equal", adjustable="box")
    ax.axis("off")

    values = frame[TRANSFERRED].to_numpy()
    missing = ~np.isfinite(values)
    ax.scatter(frame.xr[missing], frame.yr[missing], s=SPATIAL_DOT, color=NO_VALUE, linewidths=0, zorder=1)
    ax.scatter(frame.xr[~missing], frame.yr[~missing], s=SPATIAL_DOT, c=CMAP(NORM(values[~missing])), linewidths=0,
               zorder=2)
    fig.savefig(path, dpi=SPATIAL_DPI, facecolor="black")
    fig.savefig(path.with_suffix(".pdf"), dpi=SPATIAL_DPI, facecolor="black")
    fig.savefig(path.with_suffix(".svg"), dpi=SPATIAL_DPI, facecolor="black")
    plt.close(fig)


def spatial_key(path: Path) -> None:
    """Horizontal color key for the section panels."""
    fig = plt.figure(figsize=(3.0, 1.0), facecolor="black")
    cax = fig.add_axes([0.12, 0.45, 0.76, 0.2])
    bar = fig.colorbar(mpl.cm.ScalarMappable(norm=NORM, cmap=CMAP), cax=cax, orientation="horizontal",
                       ticks=[0, 0.5, 1])
    bar.ax.set_xticklabels(["0 early", "0.5", "1 late"], color=TEXT, fontsize=8)
    bar.outline.set_visible(False)
    bar.ax.tick_params(color=TEXT, length=2)
    cax.set_title("transferred pseudotime (rank)", color=TEXT, fontsize=8, pad=4)
    for ext in ("png", "pdf", "svg"):
        fig.savefig(path.with_suffix(f".{ext}"), dpi=300, facecolor="black")
    plt.close(fig)

## Run

In [ ]:
repo = find_repo(REPO)
notebook = repo / NOTEBOOK
if not notebook.is_file():
    raise FileNotFoundError(f"{notebook} not found; set NOTEBOOK to this notebook's path")

out = repo / OUT
panel_files = {phase: out / f"{DATE}-Human-{phase}-transferred_pseudotime-spatial.png" for phase in SECTIONS}
key_file = out / f"{DATE}-spatial_key_transferred_pseudotime.png"
sidecar = out / "fig6e_sections_transferred_pseudotime.provenance.json"
outputs = (
    [path.with_suffix(ext) for path in panel_files.values() for ext in (".png", ".pdf", ".svg")]
    + [key_file.with_suffix(ext) for ext in (".png", ".pdf", ".svg")]
)
existing = [str(path) for path in outputs + [sidecar] if path.exists()]
if existing and not FORCE:
    raise FileExistsError(f"Refusing to overwrite {existing}; set FORCE = True")
out.mkdir(parents=True, exist_ok=True)

In [ ]:
joint = pd.read_parquet(repo / JOINT)
human = human_nuclei(joint, repo / HUMAN_TABLE)
# Flagged embedding outliers are drawn like the rest (1 Nov 2026); the count goes to the provenance record only.
flagged = human.embedding_outlier.astype(bool)
LOG.info("Human nuclei: %d drawn candidates, %d of them flagged embedding outliers", len(human), flagged.sum())

In [ ]:
font = set_font()
LOG.info("Font: %s", font)

counts = {}
for phase, spec in SECTIONS.items():
    cells = human[(human.timepoint.astype(str) == phase) & human.x_um_qc.notna()]
    frame = section_frame(cells, spec["limits"], spec["rotation"], spec["mirror"])
    spatial_panel(frame, panel_files[phase])
    counts[phase] = {
        "libraries": sorted(frame["sample"].astype(str).unique()),
        "drawn": len(frame),
        "no_value": int(frame[TRANSFERRED].isna().sum()),
        "median_transferred": round(float(frame[TRANSFERRED].median()), 4),
        "limits_um": spec["limits"],
        "rotation_deg": spec["rotation"],
        "mirrored_left_right": spec["mirror"],
    }
    LOG.info("%s: %s", phase, counts[phase])
spatial_key(key_file)
LOG.info("Wrote %s", [str(path) for path in outputs])

for phase, path in panel_files.items():
    display(Image(filename=str(path), width=320))

In [ ]:
parameters = {"REPO": str(repo), "JOINT": JOINT, "HUMAN_TABLE": HUMAN_TABLE, "OUT": OUT, "DATE": DATE, "FORCE": FORCE}

record = {
    "outputs": [str(path.resolve()) for path in outputs],
    "written": datetime.datetime.now().isoformat(timespec="seconds"),
    "host": platform.node(),
    "notebook": str(notebook.resolve()),
    "notebook_md5": hashlib.md5(notebook.read_bytes()).hexdigest(),
    "git": git_revision(notebook),
    "parameters": parameters,
    "inputs": [file_record(repo / JOINT, "joint"), file_record(repo / HUMAN_TABLE, "human_table")],
    "font": font,
    "versions": {"python": platform.python_version(), "matplotlib": mpl.__version__, "pandas": pd.__version__},
    "transferred_column": TRANSFERRED,
    "rescaled_or_reranked": False,
    "colormap": "viridis, fixed 0-1",
}

record["embedding_outliers_omitted"] = 0
record["embedding_outliers_drawn"] = int(flagged.sum())
record["spatial"] = counts
sidecar.write_text(json.dumps(record, indent=1, default=str))
LOG.info("Wrote %s", sidecar)